## IMPORTS

In [5]:
import pandas as pd
import numpy as np

from rdkit import Chem
from rdkit.Chem import AllChem

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

import joblib
import os

/home/muhamed/.conda/envs/ctddg_env/lib/python3.9/site-packages/sklearn/utils/fixes.py:28: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version  # type: ignore


## DEFINE DATASET PATHS

In [6]:
DATA_DIR = os.path.expanduser("/home/muhamed/repo/CTDDG/datasets")

hansen_v1_path = os.path.join(DATA_DIR, "Mutagenicity_N7090.csv")
hansen_v2_path = os.path.join(DATA_DIR, "Mutagenicity_N6512.csv")

## LOAD THE 2 HANSEN DATASETS

In [7]:
hansen_v1 = pd.read_csv(
    hansen_v1_path,
    sep="\t"
)

hansen_v2 = pd.read_csv(
    hansen_v2_path,
    sep=","
)

print("Hansen Version 1 shape:", hansen_v1.shape)
print("Hansen Version 2 shape:", hansen_v2.shape)

Hansen Version 1 shape: (7090, 7)
Hansen Version 2 shape: (6512, 7)


## BASIC DATASET INSPECTION


In [8]:
print("Version 1 columns:")
print(hansen_v1.columns.tolist())

print("\nVersion 2 columns:")
print(hansen_v2.columns.tolist())

print("\nVersion 1:")
display(hansen_v1.head())

print("\nVersion 2:")
display(hansen_v2.head())

Version 1 columns:
['CAS_NO', 'Source', 'Activity', 'Steroid', 'WDI', 'Canonical_Smiles', 'REFERENCE']

Version 2 columns:
['CAS_NO', 'Source', 'Activity', 'Steroid', 'WDI', 'Canonical_Smiles', 'REFERENCE']

Version 1:


,CAS_NO,Source,Activity,Steroid,WDI,Canonical_Smiles,REFERENCE
0,59282-34-7,CCRIS,0,0,NaN,COc1cc(c(O)c(c1)C(C)(C)C)[N+](=O)[O-],"KALUS,WH, MUENZNER,R AND FILBY,WG, ISOLATION A..."
1,594-20-7,CCRIS,0,0,NaN,CC(C)(Cl)Cl,"TORNERO-VELEZ,R ROSS,MK GRANVILLE,C LASKEY,J J..."
2,38363-41-6,CCRIS,1,0,ISOPENBUT,CC(C)(C)NCC(O)COc1ccccc1C2CCCC2,"HUSSAIN,SS, ABOUL-ENEIN,HY, AL-DAKAN,A AND HAN..."
3,38363-40-5,CCRIS,1,0,NaN,CC(C)(C)NC[C@H](O)COc1ccccc1C2CCCC2,"HUSSAIN,SS, ABOUL-ENEIN,HY, AL-DAKAN,A AND HAN..."
4,594-70-7,CCRIS,0,0,NaN,CC(C)(C)[N+](=O)[O-],"CONAWAY,CC, HUSSAIN,NS, WAY,BM AND FIALA,ES, E..."



Version 2:


,CAS_NO,Source,Activity,Steroid,WDI,Canonical_Smiles,REFERENCE
0,2475-33-4,VITIC,0,-1,NaN,O=C1c2ccccc2C(=O)c3c1ccc4c3[nH]c5c6C(=O)c7cccc...,"JUDSON, PN, COOKE, PA, DOERRER, NG, GREENE, N,..."
1,820-75-7,CCRIS,1,-1,NaN,NNC(=O)CNC(=O)\C=N\#N,"MCCANN,J, CHOI,E, YAMASAKI,E AND AMES,BN, DETE..."
2,2435-76-9,CCRIS,1,-1,NaN,O=C1NC(=O)\C(=N/#N)\C=N1,"HIRAMOTO,K, KATO,T AND KIKUGAWA,K, MECHANISMS ..."
3,817-99-2,CCRIS,1,-1,NaN,NC(=O)CNC(=O)\C=N\#N,"MCCANN,J, CHOI,E, YAMASAKI,E AND AMES,BN, DETE..."
4,116539-70-9,CCRIS,1,-1,NaN,CCCCN(CC(O)C1=C\C(=N/#N)\C(=O)C=C1)N=O,"KIKUGAWA,K, KATO,T AND TAKEDA,Y, FORMATION OF ..."


## DATASET INFORMATION

In [9]:
print("VERSION 1 INFO")
hansen_v1.info()

print("\n" + "="*60)

print("VERSION 2 INFO")
hansen_v2.info()

VERSION 1 INFO
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7090 entries, 0 to 7089
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   CAS_NO            7090 non-null   object
 1   Source            7090 non-null   object
 2   Activity          7090 non-null   int64 
 3   Steroid           7090 non-null   int64 
 4   WDI               1493 non-null   object
 5   Canonical_Smiles  7090 non-null   object
 6   REFERENCE         7090 non-null   object
dtypes: int64(2), object(5)
memory usage: 387.9+ KB

VERSION 2 INFO
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6512 entries, 0 to 6511
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   CAS_NO            6512 non-null   object
 1   Source            6512 non-null   object
 2   Activity          6512 non-null   int64 
 3   Steroid           6512 non-null   int64 
 4   WDI       

## CHECK MISSING VALUES

In [10]:
print("Missing values — Version 1")
display(hansen_v1.isnull().sum())

print("\nMissing values — Version 2")
display(hansen_v2.isnull().sum())

Missing values — Version 1


CAS_NO                 0
Source                 0
Activity               0
Steroid                0
WDI                 5597
Canonical_Smiles       0
REFERENCE              0
dtype: int64


Missing values — Version 2


CAS_NO                 0
Source                 0
Activity               0
Steroid                0
WDI                 5096
Canonical_Smiles       0
REFERENCE              0
dtype: int64

## CHECK THE TARGET DISTRIBUTION

In [11]:
print("Version 1 Activity distribution:")
print(hansen_v1["Activity"].value_counts())

print("\nVersion 2 Activity distribution:")
print(hansen_v2["Activity"].value_counts())

print("\nVersion 1 Activity proportions:")
print(hansen_v1["Activity"].value_counts(normalize=True))

print("\nVersion 2 Activity proportions:")
print(hansen_v2["Activity"].value_counts(normalize=True))

Version 1 Activity distribution:
1    3767
0    3323
Name: Activity, dtype: int64

Version 2 Activity distribution:
1    3503
0    3009
Name: Activity, dtype: int64

Version 1 Activity proportions:
1    0.531312
0    0.468688
Name: Activity, dtype: float64

Version 2 Activity proportions:
1    0.53793
0    0.46207
Name: Activity, dtype: float64


## CHECK WHETHER THE TWO VERSIONS OVERLAP

In [12]:
smiles_v1 = set(hansen_v1["Canonical_Smiles"].dropna())
smiles_v2 = set(hansen_v2["Canonical_Smiles"].dropna())

overlap = smiles_v1.intersection(smiles_v2)

print("Unique SMILES in Version 1:", len(smiles_v1))
print("Unique SMILES in Version 2:", len(smiles_v2))
print("SMILES present in both:", len(overlap))

Unique SMILES in Version 1: 7090
Unique SMILES in Version 2: 6512
SMILES present in both: 6512


## MERGE THE DATASETS

In [13]:
hansen_combined = pd.concat(
    [hansen_v1, hansen_v2],
    ignore_index=True
)

print("Combined shape before removing duplicates:")
print(hansen_combined.shape)

Combined shape before removing duplicates:
(13602, 7)


## REMOVE EXACT DUPLICATE RECORDS

In [14]:
hansen_combined = hansen_combined.drop_duplicates()

print("Shape after exact duplicate removal:")
print(hansen_combined.shape)

Shape after exact duplicate removal:
(13543, 7)


## REMOVE DUPLICATE MOLECULES

In [15]:
hansen_combined = hansen_combined.drop_duplicates(
    subset=["Canonical_Smiles"],
    keep="first"
).reset_index(drop=True)

print("Shape after unique SMILES filtering:")
print(hansen_combined.shape)

Shape after unique SMILES filtering:
(7090, 7)


## FINAL DATASET INSPECTION

In [16]:
print("Final dataset shape:", hansen_combined.shape)

print("\nColumns:")
print(hansen_combined.columns.tolist())

print("\nMissing values:")
display(hansen_combined.isnull().sum())

print("\nActivity distribution:")
print(hansen_combined["Activity"].value_counts())

display(hansen_combined.head())

Final dataset shape: (7090, 7)

Columns:
['CAS_NO', 'Source', 'Activity', 'Steroid', 'WDI', 'Canonical_Smiles', 'REFERENCE']

Missing values:


CAS_NO                 0
Source                 0
Activity               0
Steroid                0
WDI                 5597
Canonical_Smiles       0
REFERENCE              0
dtype: int64


Activity distribution:
1    3767
0    3323
Name: Activity, dtype: int64


,CAS_NO,Source,Activity,Steroid,WDI,Canonical_Smiles,REFERENCE
0,59282-34-7,CCRIS,0,0,NaN,COc1cc(c(O)c(c1)C(C)(C)C)[N+](=O)[O-],"KALUS,WH, MUENZNER,R AND FILBY,WG, ISOLATION A..."
1,594-20-7,CCRIS,0,0,NaN,CC(C)(Cl)Cl,"TORNERO-VELEZ,R ROSS,MK GRANVILLE,C LASKEY,J J..."
2,38363-41-6,CCRIS,1,0,ISOPENBUT,CC(C)(C)NCC(O)COc1ccccc1C2CCCC2,"HUSSAIN,SS, ABOUL-ENEIN,HY, AL-DAKAN,A AND HAN..."
3,38363-40-5,CCRIS,1,0,NaN,CC(C)(C)NC[C@H](O)COc1ccccc1C2CCCC2,"HUSSAIN,SS, ABOUL-ENEIN,HY, AL-DAKAN,A AND HAN..."
4,594-70-7,CCRIS,0,0,NaN,CC(C)(C)[N+](=O)[O-],"CONAWAY,CC, HUSSAIN,NS, WAY,BM AND FIALA,ES, E..."


## REMOVE RECORDS WITHOUT SMILES OR LABELS

In [17]:
hansen_combined = hansen_combined.dropna(
    subset=["Canonical_Smiles", "Activity"]
).reset_index(drop=True)

print("Shape after removing missing SMILES/Activity:")
print(hansen_combined.shape)

Shape after removing missing SMILES/Activity:
(7090, 7)


## CONVERT SMILES TO RDKIT MOLECULES

In [18]:
hansen_combined["Mol"] = hansen_combined["Canonical_Smiles"].apply(
    Chem.MolFromSmiles
)

[02:01:59] SMILES Parse Error: syntax error while parsing: O=C1NC(=O)\C(=N/#N)\C=N1
[02:01:59] SMILES Parse Error: Failed parsing SMILES 'O=C1NC(=O)\C(=N/#N)\C=N1' for input: 'O=C1NC(=O)\C(=N/#N)\C=N1'
[02:01:59] SMILES Parse Error: syntax error while parsing: NNC(=O)CNC(=O)\C=N\#N
[02:01:59] SMILES Parse Error: Failed parsing SMILES 'NNC(=O)CNC(=O)\C=N\#N' for input: 'NNC(=O)CNC(=O)\C=N\#N'
[02:01:59] SMILES Parse Error: syntax error while parsing: NC(=O)CNC(=O)\C=N\#N
[02:01:59] SMILES Parse Error: Failed parsing SMILES 'NC(=O)CNC(=O)\C=N\#N' for input: 'NC(=O)CNC(=O)\C=N\#N'
[02:01:59] SMILES Parse Error: syntax error while parsing: CCN(CC(O)C1=CC(=O)\C(=N\#N)\C=C1)N=O
[02:01:59] SMILES Parse Error: Failed parsing SMILES 'CCN(CC(O)C1=CC(=O)\C(=N\#N)\C=C1)N=O' for input: 'CCN(CC(O)C1=CC(=O)\C(=N\#N)\C=C1)N=O'
[02:01:59] Explicit valence for atom # 0 N, 5, is greater than permitted
[02:01:59] SMILES Parse Error: syntax error while parsing: NC(COC(=O)\C=N/#N)C(=O)O
[02:01:59] SMILES Pa

In [19]:
invalid_mols = hansen_combined["Mol"].isna().sum()

print("Invalid SMILES:", invalid_mols)

Invalid SMILES: 7


In [20]:
hansen_combined = hansen_combined[
    hansen_combined["Mol"].notna()
].reset_index(drop=True)

print("Final valid molecules:", len(hansen_combined))

Final valid molecules: 7083


## CONVERT ACTIVITY TO INTEGER

In [21]:
hansen_combined["Activity"] = hansen_combined["Activity"].astype(int)

print(hansen_combined["Activity"].value_counts())

1    3761
0    3322
Name: Activity, dtype: int64


## GENERATE MORGAN FINGERPRINTS

In [22]:
def generate_morgan_fingerprint(mol, radius=2, n_bits=2048):
    return AllChem.GetMorganFingerprintAsBitVect(
        mol,
        radius,
        nBits=n_bits
    )

In [23]:
fingerprints = hansen_combined["Mol"].apply(
    generate_morgan_fingerprint
)

## CONVERT FINGERPRINTS TO NUMPY MATRIX

In [24]:
X = np.array([
    np.array(fp)
    for fp in fingerprints
])

y = hansen_combined["Activity"].values

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

Feature matrix shape: (7083, 2048)
Target shape: (7083,)


## TRAIN/TEST SPLIT

In [25]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

Training samples: 5666
Testing samples: 1417


## BUILD RANDOM FOREST

In [26]:
rf_model = RandomForestClassifier(
    n_estimators=500,
    random_state=42,
    n_jobs=-1
)

In [27]:
rf_model.fit(X_train, y_train)

print("Random Forest training completed.")

Random Forest training completed.


## PREDICTIONS

In [28]:
y_pred = rf_model.predict(X_test)

y_prob = rf_model.predict_proba(X_test)[:, 1]

## EVALUATE

In [29]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1-score :", f1)
print("ROC-AUC  :", roc_auc)

Accuracy : 0.8108680310515173
Precision: 0.8315068493150685
Recall   : 0.8071808510638298
F1-score : 0.8191632928475034
ROC-AUC  : 0.8896126619740842


In [30]:
print(classification_report(
    y_test,
    y_pred,
    target_names=["Ames Negative", "Ames Positive"]
))

               precision    recall  f1-score   support

Ames Negative       0.79      0.82      0.80       665
Ames Positive       0.83      0.81      0.82       752

     accuracy                           0.81      1417
    macro avg       0.81      0.81      0.81      1417
 weighted avg       0.81      0.81      0.81      1417



## CONFUSION MATRIX

In [31]:
cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[542 123]
 [145 607]]


## SAVE THE TRAINED MODEL

In [32]:
os.makedirs(
    os.path.expanduser("/home/muhamed/repo/CTDDG/models"),
    exist_ok=True
)

In [33]:
model_path = os.path.expanduser(
    "/home/muhamed/repo/CTDDG/models/ames_random_forest.pkl"
)

joblib.dump(rf_model, model_path)

print("Model saved to:")
print(model_path)

Model saved to:
/home/muhamed/repo/CTDDG/models/ames_random_forest.pkl


## SAVE THE FINGERPRINT CONFIGURATION

In [34]:
fingerprint_config = {
    "type": "Morgan",
    "radius": 2,
    "n_bits": 2048
}

config_path = os.path.expanduser(
    "/home/muhamed/repo/CTDDG/models/fingerprint_config.pkl"
)

joblib.dump(fingerprint_config, config_path)

print("Fingerprint configuration saved.")

Fingerprint configuration saved.


# ANNOTATION OF CHEMBL DATASET

In [35]:
import pandas as pd
import os

chembl_path = "/home/muhamed/repo/CTDDG/datasets/data/chembl/chembl.csv"

chembl_df = pd.read_csv(chembl_path,sep=";",quotechar='"')

print("ChEMBL shape:", chembl_df.shape)
print("Columns:")
print(chembl_df.columns.tolist())

print(chembl_df.head())

ChEMBL shape: (1207360, 31)
Columns:
['ChEMBL ID', 'Name', 'Synonyms', 'Type', 'Max Phase', 'Molecular Weight', 'Targets', 'Bioactivities', 'AlogP', 'Polar Surface Area', 'HBA', 'HBD', '#RO5 Violations', '#Rotatable Bonds', 'Passes Ro3', 'QED Weighted', 'CX Acidic pKa', 'CX Basic pKa', 'CX LogP', 'CX LogD', 'Aromatic Rings', 'Structure Type', 'Inorganic Flag', 'Heavy Atoms', 'HBA (Lipinski)', 'HBD (Lipinski)', '#RO5 Violations (Lipinski)', 'Molecular Weight (Monoisotopic)', 'Molecular Species', 'Molecular Formula', 'Smiles']
       ChEMBL ID Name     Synonyms            Type  Max Phase  \
0  CHEMBL1902090  NaN          NaN  Small molecule          0   
1  CHEMBL1802565  NaN          NaN  Small molecule          0   
2  CHEMBL1531434  NaN          NaN  Small molecule          0   
3  CHEMBL3304157  NaN          NaN  Small molecule          0   
4   CHEMBL592102  NaN  SJ000185809  Small molecule          0   

   Molecular Weight  Targets  Bioactivities  AlogP  Polar Surface Area  ...  \

## Convert ChEMBL SMILES to RDKit molecules

In [36]:
from rdkit import Chem

chembl_df["Mol"] = chembl_df["Smiles"].apply(Chem.MolFromSmiles)

invalid_mask = chembl_df["Mol"].isna()

print("Total ChEMBL compounds:", len(chembl_df))
print("Valid SMILES:", (~invalid_mask).sum())
print("Invalid SMILES:", invalid_mask.sum())

[02:02:45] Can't kekulize mol.  Unkekulized atoms: 1 2 3 4 5 6 10 11 15 16 17 19 20 21
[02:03:36] WARNING: not removing hydrogen atom without neighbors
[02:04:11] WARNING: not removing hydrogen atom without neighbors


Total ChEMBL compounds: 1207360
Valid SMILES: 1207359
Invalid SMILES: 1


## Generate Morgan fingerprints

In [37]:
from rdkit.Chem import AllChem

def mol_to_morgan(mol):
    if mol is None:
        return None
    
    return AllChem.GetMorganFingerprintAsBitVect(
        mol,
        radius=fingerprint_config["radius"],
        nBits=fingerprint_config["n_bits"]
    )

chembl_df["Fingerprint"] = chembl_df["Mol"].apply(mol_to_morgan)

print("Fingerprints generated.")

Fingerprints generated.


## Creating  the ChEMBL feature matrix

In [41]:
import numpy as np
from rdkit import DataStructs

batch_size = 50_000

ames_predictions = np.full(len(chembl_df), np.nan)

for start in range(0, len(chembl_df), batch_size):

    end = min(start + batch_size, len(chembl_df))

    batch = chembl_df.iloc[start:end]

    valid = batch["Fingerprint"].notna()
    fps = batch.loc[valid, "Fingerprint"]

    X_batch = np.zeros(
        (len(fps), 2048),
        dtype=np.uint8
    )

    for i, fp in enumerate(fps):
        DataStructs.ConvertToNumpyArray(fp, X_batch[i])

    if len(fps) > 0:

        predictions = rf_model.predict(X_batch)

        valid_indices = np.where(valid.values)[0]

        ames_predictions[start + valid_indices] = predictions

    print(f"Processed {end:,} / {len(chembl_df):,}")

chembl_df["Ames_prediction"] = ames_predictions

print(chembl_df["Ames_prediction"].value_counts(dropna=False))

Processed 50,000 / 1,207,360
Processed 100,000 / 1,207,360
Processed 150,000 / 1,207,360
Processed 200,000 / 1,207,360
Processed 250,000 / 1,207,360
Processed 300,000 / 1,207,360
Processed 350,000 / 1,207,360
Processed 400,000 / 1,207,360
Processed 450,000 / 1,207,360
Processed 500,000 / 1,207,360
Processed 550,000 / 1,207,360
Processed 600,000 / 1,207,360
Processed 650,000 / 1,207,360
Processed 700,000 / 1,207,360
Processed 750,000 / 1,207,360
Processed 800,000 / 1,207,360
Processed 850,000 / 1,207,360
Processed 900,000 / 1,207,360
Processed 950,000 / 1,207,360
Processed 1,000,000 / 1,207,360
Processed 1,050,000 / 1,207,360
Processed 1,100,000 / 1,207,360
Processed 1,150,000 / 1,207,360
Processed 1,200,000 / 1,207,360
Processed 1,207,360 / 1,207,360
0.0    919739
1.0    287620
NaN         1
Name: Ames_prediction, dtype: int64


In [42]:
print(chembl_df.shape)

print("\nColumns:")
print(chembl_df.columns.tolist())

print("\nAmes prediction counts:")
print(chembl_df["Ames_prediction"].value_counts(dropna=False))

(1207360, 34)

Columns:
['ChEMBL ID', 'Name', 'Synonyms', 'Type', 'Max Phase', 'Molecular Weight', 'Targets', 'Bioactivities', 'AlogP', 'Polar Surface Area', 'HBA', 'HBD', '#RO5 Violations', '#Rotatable Bonds', 'Passes Ro3', 'QED Weighted', 'CX Acidic pKa', 'CX Basic pKa', 'CX LogP', 'CX LogD', 'Aromatic Rings', 'Structure Type', 'Inorganic Flag', 'Heavy Atoms', 'HBA (Lipinski)', 'HBD (Lipinski)', '#RO5 Violations (Lipinski)', 'Molecular Weight (Monoisotopic)', 'Molecular Species', 'Molecular Formula', 'Smiles', 'Mol', 'Fingerprint', 'Ames_prediction']

Ames prediction counts:
0.0    919739
1.0    287620
NaN         1
Name: Ames_prediction, dtype: int64


In [43]:
chembl_df[["Ames_prediction"]].head(10)

,Ames_prediction
0,1.0
1,0.0
2,0.0
3,0.0
4,0.0
5,0.0
6,0.0
7,0.0
8,0.0
9,0.0


In [47]:
import os

output_dir = "/home/muhamed/repo/CTDDG/datasets/data/chembl"

os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(
    output_dir,
    "chembl_with_ames_predictions.csv"
)

chembl_df.to_csv(output_path, index=False)

print(f"Saved successfully to: {output_path}")

Saved successfully to: /home/muhamed/repo/CTDDG/datasets/data/chembl/chembl_with_ames_predictions.csv


In [48]:
import os

print("File exists:", os.path.exists(output_path))
print("File size:", os.path.getsize(output_path) / (1024**3), "GB")

File exists: True
File size: 0.35155916120857 GB


In [ ]:
test_df = pd.read_csv(output_path)

print("Loaded shape:", test_df.shape)
print("Ames predictions:")
print(test_df["Ames_prediction"].value_counts(dropna=False))

Loaded shape: (1207360, 34)
Ames predictions:
0.0    919739
1.0    287620
NaN         1
Name: Ames_prediction, dtype: int64


In [51]:
test_df.head(7)

,ChEMBL ID,Name,Synonyms,Type,Max Phase,Molecular Weight,Targets,Bioactivities,AlogP,Polar Surface Area,...,HBA (Lipinski),HBD (Lipinski),#RO5 Violations (Lipinski),Molecular Weight (Monoisotopic),Molecular Species,Molecular Formula,Smiles,Mol,Fingerprint,Ames_prediction
0,CHEMBL1902090,NaN,NaN,Small molecule,0,337.42,2.0,2.0,2.10,62.30,...,5,1,0,337.1790,NEUTRAL,C20H23N3O2,Cc1ccc(CCN2CC(C(=O)NCc3cccnc3)CC2=O)cc1,<rdkit.Chem.rdchem.Mol object at 0x7f5740aaf580>,<rdkit.DataStructs.cDataStructs.ExplicitBitVec...,1.0
1,CHEMBL1802565,NaN,NaN,Small molecule,0,336.34,17.0,26.0,2.12,75.71,...,6,1,0,336.0580,NEUTRAL,C15H13FN2O4S,O=C1NCCN1c1ccc(S(=O)(=O)Oc2ccc(F)cc2)cc1,<rdkit.Chem.rdchem.Mol object at 0x7f573b45ff20>,<rdkit.DataStructs.cDataStructs.ExplicitBitVec...,0.0
2,CHEMBL1531434,NaN,NaN,Small molecule,0,440.52,4.0,4.0,4.48,80.49,...,7,1,0,440.2224,NEUTRAL,C24H29FN4O3,CCOC(=O)c1c(C)oc2nc(C)nc(NCC3CCN(Cc4ccc(F)cc4)...,<rdkit.Chem.rdchem.Mol object at 0x7f573b45fd60>,<rdkit.DataStructs.cDataStructs.ExplicitBitVec...,0.0
3,CHEMBL3304157,NaN,NaN,Small molecule,0,148.21,NaN,NaN,1.09,17.29,...,2,0,0,148.1000,BASE,C9H12N2,Cn1cccc/c1=N\C1CC1,<rdkit.Chem.rdchem.Mol object at 0x7f573b45fe40>,<rdkit.DataStructs.cDataStructs.ExplicitBitVec...,0.0
4,CHEMBL592102,NaN,SJ000185809,Small molecule,0,306.48,1.0,2.0,2.48,30.54,...,4,2,0,306.1878,NEUTRAL,C16H26N4S,CCCCc1ccc(NC(=S)NN2CCN(C)CC2)cc1,<rdkit.Chem.rdchem.Mol object at 0x7f573b45feb0>,<rdkit.DataStructs.cDataStructs.ExplicitBitVec...,0.0
5,CHEMBL3342339,NaN,NaN,Small molecule,0,395.42,10.0,21.0,1.35,95.68,...,6,2,0,395.1270,NEUTRAL,C24H17N3O3,O=C(c1ccccc1)c1cccc(/C=c2\[nH]c(=O)/c(=C/c3ccc...,<rdkit.Chem.rdchem.Mol object at 0x7f573b45fc80>,<rdkit.DataStructs.cDataStructs.ExplicitBitVec...,0.0
6,CHEMBL1624292,NaN,NaN,Small molecule,0,323.48,NaN,NaN,4.69,12.47,...,2,0,0,323.2249,BASE,C22H29NO,CC1(C)CC(N(CCc2ccccc2)Cc2ccccc2)CCO1,<rdkit.Chem.rdchem.Mol object at 0x7f573b45fc10>,<rdkit.DataStructs.cDataStructs.ExplicitBitVec...,0.0


In [1]:
import os

output_dir = "/home/muhamed/repo/CTDDG/datasets/data/chembl"

os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(
    output_dir,
    "chembl_with_ames_predictions.csv"
)

In [4]:
import pandas as pd

test_df = pd.read_csv(output_path)

print("Loaded shape:", test_df.shape)
print("Ames predictions:")
print(test_df["Ames_prediction"].value_counts(dropna=False))

Loaded shape: (1207360, 34)
Ames predictions:
0.0    919739
1.0    287620
NaN         1
Name: Ames_prediction, dtype: int64


In [5]:
test_df = test_df.dropna(subset=["Ames_prediction"]).copy()

In [7]:
print(test_df.shape)
print(test_df["Ames_prediction"].value_counts(dropna=False))

(1207359, 34)
0.0    919739
1.0    287620
Name: Ames_prediction, dtype: int64


In [8]:
test_df.to_csv(output_path, index=False)